<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_03_thermal.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.1 · Notebook 03 — thermal coupling

**Paired with L10.1 · Battery models**

Add the energy equation with the Gu & Wang heat source (Eq. 25), **including**
the entropic term the original paper had to neglect.

$$q = \underbrace{a_s i_n \eta}_{\text{irreversible}}
+ \underbrace{a_s i_n T \frac{\partial U}{\partial T}}_{\text{reversible}}
+ \underbrace{\sigma|\nabla\phi_s|^2
+ \kappa|\nabla\phi_e|^2}_{\text{ohmic}}$$

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## TODO 1 — the three heat terms, computed separately

In [ ]:
cell = pb.CellParams(c_rate=2.0, T_amb=298.15, h_cool=10.0)
print(cell, "\n  Biot =", round(cell.biot, 4),
      "->", "lumped defensible" if cell.biot < 0.1 else "gradient matters")

# Representative values; replace with your trained fields.
a_s, i_n = 3.8e5, 20.0          # 1/m, A/m2
eta      = 0.03                  # V
dUdT     = -1.0e-4               # V/K  (ORegan2022 measures this properly)
q_irr, q_rev, q_ohm = pb.heat_source_terms(i_n, a_s, eta, cell.T_amb, dUdT)
for nm, q in [("irreversible", q_irr), ("reversible", q_rev), ("ohmic", q_ohm)]:
    print(f"  {nm:14s} {q:12.3e} W/m3")
print(f"  {'total':14s} {q_irr + q_rev + q_ohm:12.3e} W/m3")

**Note the sign.** The reversible term follows the current direction, so it
changes sign between charge and discharge — the only one of the three that
does. Reverse `i_n` and confirm.

## TODO 2 — the temperature response

In [ ]:
q_tot = q_irr + q_rev + q_ohm
t = np.linspace(0, cell.t_discharge, 400)
T = pb.lumped_temperature(cell, q_tot, t)
plt.figure(figsize=(6.5, 3.2))
plt.plot(t / 60, T - 273.15)
plt.xlabel("time [min]"); plt.ylabel("cell temperature [degC]")
plt.tight_layout(); plt.show()
print(f"temperature rise: {T[-1]-cell.T_amb:.1f} K")

## TODO 3 — the r–z field

In [ ]:
pb.plot_rz_field(cell, T_core_rise=float(T[-1] - cell.T_amb))

## TODO 4 — coupled versus decoupled

Recompute with `pb.arrhenius` applied to the diffusivity, and again with the
property frozen at its reference value. Gu & Wang report that the decoupled
model **underpredicts voltage and overpredicts temperature**. Do you reproduce
the direction of both errors?

In [ ]:
for T_test in (298.15, 313.15, 333.15):
    Ds = pb.arrhenius(cell.Ds_n, cell.E_act_Ds, T_test)
    print(f"  T = {T_test-273.15:5.1f} C   Ds = {Ds:.3e} m2/s   "
          f"({Ds/cell.Ds_n:.2f}x reference)")